# 01 — Reproducible data preparation and audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oxedanda/avcad_final_project/blob/main/notebooks/01_data_loading_and_audit.ipynb)

This notebook rebuilds every processed CSV directly from the original INE `.xls` exports. It replaces the earlier manually typed DataFrames and preserves missing values without shifting neighbouring crop columns.

In [ ]:
from pathlib import Path

REPO_DIR = Path('/content/avcad_final_project')
if REPO_DIR.exists():
    %cd /content/avcad_final_project
    !git pull --ff-only
else:
    !git clone https://github.com/oxedanda/avcad_final_project.git /content/avcad_final_project
    %cd /content/avcad_final_project
!pip -q install -r requirements.txt

## Extract the INE tables

The parser reads the `Quadro` sheet, resolves merged multi-row headers, selects `Continente` (code 1), converts the INE missing markers (`x`, `-`) to `NaN`, translates labels, and checks the four target years.

In [ ]:
from IPython.display import display
from src.prepare_data import prepare_processed_data

frames = prepare_processed_data()
for filename, frame in frames.items():
    print(f'\n{filename}: {frame.shape[0]} rows × {frame.shape[1]} columns')
    display(frame)

## Audit traceability and missingness

In [ ]:
import pandas as pd

display(pd.read_csv('outputs/tables/data_provenance.csv'))
missing = pd.concat(
    {name: frame.isna().sum() for name, frame in frames.items()},
    axis=1,
).fillna(0).astype(int)
display(missing.loc[missing.sum(axis=1).gt(0)])

## Handoff

The validated CSVs under `data/processed/` are consumed by `src/analysis.py` and notebook 02. Running the analysis also invokes this preparation step, so the project is reproducible end to end from the raw exports.